In [5]:
import pandas as pd; 
df = pd.read_excel('ket_qua_nop_bai.xlsx'); 
print('Rows:', len(df)); print('\nColumns:', df.columns.tolist()); 
print('\nFirst 3 input_text:'); print(df['input_text'].head(3).tolist())

Rows: 500

Columns: ['raw_title', 'raw_comment', 'label_final', 'source_platform', 'likes', 'replies_count', 'id', 'input_text', 'cleaned_comment', 'cleaned_title', 'post_id', 'timestamp', 'username', 'char_length', 'word_count', 'has_emoji', 'topic', 'is_hate_keyword', 'sampling_strategy', 'label_studio_text', 'video_id']

First 3 input_text:
['vụ xe phân khối lớn đâm vào đôi bạn trẻ trên phố <person> video hành trình ghi lại cảnh chiếc bmw phóng vơi tốc độ kinh hoàng‼ nghe tiếng người bạn trai ôm bạn gái bị đứt lìa chân khóc thảm. </s> <person> thề chứ địt con mẹ tao cực ghéc mấy thể loại chạy pkl thể hiện như này địt con mẹ bảo sao', 'boy phố "gãy cánh" ngay trước mặt </s> địt mẹ sao không đạp cho mấy đạp nhỉ,', 'con thầy vợ bạn gái coser </s> nhưng mà cái vấn đề ở đây là ngoại trừ cái bạn bồ của thằng triết dái thì đứa nào trong câu chuyện này cũng ngân đù hết']


In [8]:
import pandas as pd; 
df = pd.read_csv('labeling_task_Thien.csv'); 
print('Sample 1:'); print('ORIGINAL:', df['input_text_original'].iloc[0][:150]); 
print('CLEANED: ', df['input_text'].iloc[0][:150]); print('\nSample 2:'); 
print('ORIGINAL:', df['input_text_original'].iloc[1][:150]); print('CLEANED: ', df['input_text'].iloc[1][:150])

Sample 1:


KeyError: 'input_text_original'

In [10]:
!pip install underthesea

In [1]:
"""
🔥 FIXED NER MASKING - Xử lý đúng format của underthesea.ner()
"""

import re

# =====================================================
# 1. DANH XƯNG + HỌ VIỆT NAM
# =====================================================

TITLES = [
    'bà', 'ông', 'anh', 'chị', 'em', 'thằng', 'con', 
    'thầy', 'cô', 'bé', 'cụ', 'bác', 'dì', 'chú',
    'youtuber', 'tiktoker', 'streamer', 'tên', 'ca sĩ',
    'diễn viên', 'mc', 'hot boy', 'hot girl'
]

VIETNAMESE_SURNAMES = {
    'nguyễn', 'trần', 'lê', 'phạm', 'hoàng', 'huỳnh', 
    'phan', 'vũ', 'võ', 'đặng', 'bùi', 'đỗ', 'hồ', 'ngô', 
    'dương', 'lý', 'lưu', 'trịnh', 'đinh', 'cao', 'tạ',
    'đào', 'vương', 'mai', 'tô', 'lâm', 'quách'
}

STOP_WORDS = {
    'ở', 'đi', 'đến', 'là', 'của', 'và', 'có', 'không',
    'thì', 'nhưng', 'mà', 'với', 'cho', 'nên', 'vì'
}

def is_likely_name(word: str) -> bool:
    """Check xem từ có khả năng là tên không"""
    if not word or len(word) < 2:
        return False
    
    word_lower = word.lower()
    
    # Check stop words
    if word_lower in STOP_WORDS:
        return False
    
    # Check họ Việt Nam
    if word_lower in VIETNAMESE_SURNAMES:
        return True
    
    # Check chữ cái đầu viết hoa (đã được capitalize)
    if len(word) > 1 and word[0].isupper():
        return True
    
    return False

def smart_capitalize_titles(text: str) -> str:
    """
    Viết hoa TÊN sau danh xưng (FIX: không bỏ sót từ)
    """
    words = text.split()
    result = []
    
    i = 0
    while i < len(words):
        word = words[i]
        
        # Nếu là danh xưng và còn từ phía sau
        if word.lower() in TITLES and i + 1 < len(words):
            result.append(word)  # Thêm danh xưng
            i += 1
            
            # Check từ tiếp theo
            if i < len(words):
                next_word = words[i]
                if is_likely_name(next_word):
                    result.append(next_word.capitalize())
                    i += 1
                    
                    # Viết hoa từ thứ 2 nếu có (vd: Nhân Vlog)
                    if i < len(words) and is_likely_name(words[i]):
                        result.append(words[i].capitalize())
                        i += 1
                else:
                    result.append(next_word)
                    i += 1
        else:
            result.append(word)
            i += 1
    
    return ' '.join(result)

def advanced_ner_masking_v3(text: str) -> str:
    """
    Version 3: FIX unpacking error
    """
    try:
        from underthesea import ner
        
        # Bước 1: Boost text
        boosted_text = smart_capitalize_titles(text)
        
        # Bước 2: Chạy NER
        entities = ner(boosted_text)
        
        # Bước 3: Parse entities (XỬ LÝ NHIỀU FORMAT)
        result_tokens = []
        
        for item in entities:
            # Xử lý format khác nhau của underthesea
            if len(item) == 3:
                word, pos, tag = item
            elif len(item) == 2:
                word, tag = item
            else:
                # Format không rõ, giữ nguyên
                result_tokens.append(str(item))
                continue
            
            # Check nếu là tên người
            if 'PER' in str(tag):
                result_tokens.append('<PERSON>')
            else:
                result_tokens.append(word.lower())
        
        # Bước 4: Gộp <PERSON> liên tiếp
        result = ' '.join(result_tokens)
        result = re.sub(r'(<PERSON>\s*)+', '<PERSON> ', result)
        
        # Bước 5: Bổ sung Regex
        result = mask_names_by_regex(result)
        
        return result.strip()
        
    except ImportError:
        # Nếu không có underthesea, dùng rule-based
        print("⚠️  underthesea not found, using rule-based only")
        return rule_based_name_masking(text)
    except Exception as e:
        # Nếu NER lỗi bất kỳ, fallback về rule-based
        print(f"⚠️  NER error: {e}, using rule-based")
        return rule_based_name_masking(text)

def mask_names_by_regex(text: str) -> str:
    """
    Regex-based name detection (SIMPLIFIED - tránh catastrophic backtracking)
    """
    # Pattern 1: Họ Việt Nam + tên (đơn giản hóa)
    for surname in VIETNAMESE_SURNAMES:
        # Họ + 1-2 tên viết hoa
        pattern = rf'\b{surname}\s+[A-ZÀÁẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬĐÈÉẺẼẸÊẾỀỂỄỆÌÍỈĨỊÒÓỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÙÚỦŨỤƯỨỪỬỮỰỲÝỶỸỴ]\w+(?:\s+[A-ZÀÁẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬĐÈÉẺẼẸÊẾỀỂỄỆÌÍỈĨỊÒÓỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÙÚỦŨỤƯỨỪỬỮỰỲÝỶỸỴ]\w+)?\b'
        text = re.sub(pattern, '<PERSON>', text, flags=re.IGNORECASE)
    
    # Pattern 2: Tên sau danh xưng
    for title in TITLES:
        # Danh xưng + tên viết hoa
        pattern = rf'\b{title}\s+[A-ZÀÁẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬĐÈÉẺẼẸÊẾỀỂỄỆÌÍỈĨỊÒÓỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÙÚỦŨỤƯỨỪỬỮỰỲÝỶỸỴ]\w+(?:\s+[A-ZÀÁẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬĐÈÉẺẼẸÊẾỀỂỄỆÌÍỈĨỊÒÓỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÙÚỦŨỤƯỨỪỬỮỰỲÝỶỸỴ]\w+)?\b'
        text = re.sub(pattern, f'{title} <PERSON>', text, flags=re.IGNORECASE)
    
    # Gộp <PERSON> liên tiếp
    text = re.sub(r'(<PERSON>\s*)+', '<PERSON> ', text)
    
    return text

def rule_based_name_masking(text: str) -> str:
    """
    KHUYẾN NGHỊ: Dùng approach này (không cần underthesea)
    """
    # Bước 1: Viết hoa tên sau danh xưng
    text = smart_capitalize_titles(text)
    
    # Bước 2: Apply regex patterns
    text = mask_names_by_regex(text)
    
    # Bước 3: Lowercase về (trừ <PERSON>)
    words = text.split()
    result = []
    for word in words:
        if word == '<PERSON>':
            result.append(word)
        else:
            result.append(word.lower())
    
    return ' '.join(result)

# =====================================================
# 3. TEST
# =====================================================

if __name__ == "__main__":
    print("="*100)
    print("🔥 NER MASKING - FIXED VERSION")
    print("="*100)
    
    test_data = [
        "bà nhân vlog đi từ thiện nhưng lại quay phim", 
        "hải đăng đug jui đó",      
        "Hải Đăng đug jui",                   
        "Nguyễn Văn A và bà nhân vlog đang cãi nhau",   
        "Thạch Trang là nàng thơ du học sinh Đức",      
        "Bộ Mặt Thật của Tàng Keng Ông Trùm",
        "thằng tèo và con bé lan đi chơi",
        "anh Nam ở Hà Nội gặp chị Hoa",
        "tiktoker Khá Bảnh bị bắt",
        "ca sĩ Mỹ Tâm hát hay",
        "Nhân Phẩm Mình Bị Bôi Nhọ, Tất Nhiên Mình Không Để Yên",
        "tên là Nguyễn Văn B đấy",
        "Nguyễn Văn A nói rằng Hoàng Sa là của VN",
        "Cặp đôi Ninh - Dương gây sốt khi bán 2,8tr/vé cho fan meeting.",
    ]
    
    print(f"\n{'INPUT':<55} | {'RULE-BASED OUTPUT'}")
    print("-"*100)
    
    for text in test_data:
        result = rule_based_name_masking(text)
        print(f"{text:<55} | {result}")
    
    print("\n" + "="*100)
    print("✅ TESTED & WORKING")
    print("💡 Nếu vẫn muốn dùng NER, chạy: advanced_ner_masking_v3(text)")
    print("   Nhưng rule_based_name_masking(text) đã đủ tốt!")
    print("="*100)
    
    # Test thử NER nếu có
    print("\n--- Testing NER version (nếu có underthesea) ---")
    try:
        from underthesea import ner
        test_text = "bà nhân vlog và Nguyễn Văn A"
        result = advanced_ner_masking_v3(test_text)
        print(f"NER result: {result}")
    except Exception as e:
        print(f"NER không khả dụng: {e}")

🔥 NER MASKING - FIXED VERSION

INPUT                                                   | RULE-BASED OUTPUT
----------------------------------------------------------------------------------------------------
bà nhân vlog đi từ thiện nhưng lại quay phim            | bà <PERSON> đi từ thiện nhưng lại quay phim
hải đăng đug jui đó                                     | hải đăng đug jui đó
Hải Đăng đug jui                                        | hải đăng đug jui
Nguyễn Văn A và bà nhân vlog đang cãi nhau              | <PERSON> a và bà <PERSON> đang cãi nhau
Thạch Trang là nàng thơ du học sinh Đức                 | thạch trang là nàng thơ du học sinh đức
Bộ Mặt Thật của Tàng Keng Ông Trùm                      | bộ mặt thật của tàng keng ông <PERSON>
thằng tèo và con bé lan đi chơi                         | thằng <PERSON> con <PERSON> đi chơi
anh Nam ở Hà Nội gặp chị Hoa                            | anh <PERSON> ở hà nội gặp chị <PERSON>
tiktoker Khá Bảnh bị bắt                             

c:\Users\tran thien\.conda\envs\linear_regression\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NER result: ('bà', 'Nc', 'B-NP', 'O') ('nhân vlog', 'V', 'B-VP', 'O') ('và', 'C', 'O', 'O') ('<PERSON> A', 'Np', 'B-NP', 'B-PER')
